In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# First-Order ODEs

A first-order ODE can be written
\begin{align}
    \frac{dy}{dx} &= f(x, y)
\end{align}
and the solution is a function $y(x)$ that depends on the independent variable $x$.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE is an equation that specifies the slope of a function y(x)
    in terms of
     - the value of the function itself, y
     - the value of the variable it depends on, x.
    
    The generic form for an ODE is
    
    dy
    --  = f(x, y)
    dx
    
    '''

    def f(self, x, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, x, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')
    
    def plot_slope_field(self, ax, x, y, color='black', zorder=-1, **kwargs):
        X, Y = np.meshgrid(x, y)

        # tan(theta) = slope
        # theta = arctan(slope)
        THETA = np.arctan(self.f(X, Y))
        U = np.cos(THETA)
        V = np.sin(THETA)
    
        ax.quiver(X, Y, U, V, color=color, angles='xy', pivot='middle', zorder=zorder, **kwargs)

## The Exponential ODE

The *exponential* differential equation is given by $f(x, y) = k y$ for a real constant $k$.

It has an *exact solution*.  An *exact solution* is a *formula* for a function that when plugged into the ODE yields a true equation.  The exponential ODE is solved by
\begin{align}
    y(x) = c e^{k x}
\end{align}
for any real constant multiplier $c$.  The $k$ that appears in the solution is the $k$ in the differential equation.

In [ ]:
class ExponentialODE(FirstOrderODE):
    '''
    The exponential ODE is a first-order ODE which has f(x, y) = k y
    '''

    def __init__(self, k):
        self.k = k

    def f(self, x, y):
        return self.k * y

    def exact_solution(self, x, c):
        return c * np.exp(self.k * x)

Let's see that the exact solutions follow the slope field.

In [ ]:
k = -0.5
exp = ExponentialODE(k)

fig, ax = plt.subplots()
t = np.linspace(-4, 4, 20)
x = np.linspace(-4, 4, 20)
exp.plot_slope_field(ax, t, x)

x = np.linspace(-4, 4, 1000)
for c in (-8, -4, -2, -1, 0, 1, 2, 4, 8):
    ax.plot(x, exp.exact_solution(x, c), label=f'{c=}')

ax.set_ylim(-4,4)
ax.legend()

# *Integrating* an ODE

ODEs don't always have simple exact solutions.  But that doesn't mean there aren't any integral curves!  Instead we can think about how to find the solutions numerically.

Finding a solution to an ODE means finding a function whose slope is equal to $f$ evaluated at all the places on the curve.

The jargon is that we *integrate* the ODE---even if we're not "really" doing an antiderivative.  A tool that produces a solution to an ODE from a given starting point $(x_0, y_0)$ is *an integrator*.

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, x_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

We saw *Euler's method* for producing a solution numerically.

The *idea* of Euler's method is that $dy/dx = f(x, y)$ tells us the slope everywhere and we can approximate $y(x)$ as a sequence of little line segments whose slope matches $f(x, y)$ at the left end of each little line segment.  If you make the line segments really short you hardly make a mistake (the mistakes are like $\Delta x^2$).

Euler observed that one way to understand what a Taylor series means is that if you take a very small step the function you're expanding doesn't change that much,
\begin{align}
    y(x + \Delta x) &\approx y(x) + y'(x) \Delta x + \cdots
\end{align}
Ignoring the higher-order terms, this leads us to
\begin{align}
    \Delta y = y(x+\Delta x) - y(x) &\approx f(y, x) \Delta x
\end{align}

Suppose we start with $(x_0, y_0)$.  We can evaluate $f(y_0, x_0)$.  If you pick a small $\Delta x$ you receive a small $\Delta y$.
Call $(x_1 = x_0 + \Delta x, y_1 = y_0 + \Delta y)$.  We're walking along a little line segment whose slope is $f(y_0, x_0)$ and whose run is $\Delta x$ and therefore whose rise $\Delta y = f(y_0, x_0) \Delta x$.

In [ ]:
class EulerMethod(FirstOrderODEIntegrator):
    '''
    Euler's method is based on the Taylor series approximation

    y(x + Δx) = y(x) + y'(x) Δx + (higher-order terms with more powers of Δx which we neglect)

    Given a starting location on the solution (x_0, y_0) we can repeatedly say
    - the next x will be x_0 + Δx
    - the next y will be y_0 + Δy, and using the Taylor series Δy = y'(x) Δx
    
    This produces the next point on the solution, and we can begin again.
    '''

    def __init__(self, ODE, Delta_x, steps):

        self.Delta_x = Delta_x
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, x_0, y_0):

        # Populate all the x values
        x = x_0 + self.Delta_x * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute
        y = np.zeros_like(x, dtype=float)

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the Euler procedure over and over again.
        for step in range(self.steps):
            slope = self.ODE.f(x[step], y[step])
            y[step+1] = y[step] + slope * self.Delta_x

        return x, y

In [ ]:
k=1.0
exp = ExponentialODE(k)
exp_euler = EulerMethod(exp, 0.001, 10000)
# with Delta_x = 0.001 and 10000 steps we'll always get a total change in x of 0.001 * 10000 = 10.

x, y = exp_euler.starting_from(0, 1)
plt.plot(x, y)

x, y = exp_euler.starting_from(0, 0.5)
plt.plot(x, y)

x, y = exp_euler.starting_from(0, 2)
plt.plot(x, y)

plt.yscale('log')

# The Logistic Curve

The [*logistic function*](https://en.wikipedia.org/wiki/Logistic_function), *logistic curve*, or *sigmoid* obeys a different kind of first-order differential equation,
\begin{align}
    f(x, y) &= \frac{k}{L} y(x) \left(L-y(x) \right).
\end{align}

When $y \ll L$ the right-hand side is approximately $f(x, y) \approx \frac{k}{L} y(t) \times L = k y$, and so when $y$ is small we should expect the behavior to be approximately exponential.

When $y$ is near $L$, $y \approx L$, define $y(x) = L - u(x)$ and the ODE becomes
\begin{align}
    - \frac{du}{dx} = \frac{d}{dx}\left(L-u(x)\right) &= \frac{k}{L} (L-u(x)) (L - (L-u(x))) = \frac{k}{L} (L-u(x)) u(x)
\end{align}
which simplifies to
\begin{align}
    \frac{du}{dx} &= - \frac{k}{L} u(x) (L-u(x)).
\end{align}
and so when $u$ you get a differential equation with $f(u, x) = - k u$, so that $u$ exponentially decays and $y$ exponentially approaches $L$.

In fact, the logistic ODE has an exact solution too,
\begin{align}
    y(x) &= \frac{L}{1 + e^{-k(x-x_0)}}
\end{align}
for any real offset $x_0$ (and $L$ and $k$ are the parameters of the ODE).

In [ ]:
class LogisticODE(FirstOrderODE):
    '''
    The logistic ODE is a first-order ODE which has f(x, y) = k/L y (L - y).
    '''

    def __init__(self, k, L):
        self.k = k
        self.L = L

    def f(self, x, y):
        return self.k / self.L * y * (self.L-y)

    def exact_solution(self, x, x_0=0):
        return self.L / (1 + np.exp(-self.k * (x-x_0)))

k, L = 1, 2
logistic = LogisticODE(k, L)

x = np.linspace(-4, 4, 20)
y = np.linspace(-0.5*L, 1.5*L, 20)
fig, ax = plt.subplots()
logistic.plot_slope_field(ax, x, y)

x = np.linspace(-4, 4, 1000)
for x_0 in (-4, -2, -1, 0, 1, 2, 4):
    ax.plot(x, logistic.exact_solution(x, x_0), label=f'{x_0=}')

ax.legend()
ax.set_xlabel('x')
ax.set_ylabel('y')

But what if we didn't know the exact solution?  It's sort of a tricky formula, I wonder how someone found it.  If we were studying ODEs from scratch maybe we wouldn't have found it.  Then what could we do?

We can use the integrator with a very small $\Delta x$ and look at solutions that start at any which place.

In [ ]:
Delta_x = 0.001
steps = 10000
logistic_fine   = EulerMethod(logistic, Delta_x, steps)

fig, ax = plt.subplots()

x = np.linspace(-6, 10, 20)
y = np.linspace(0,  2,  20)
logistic.plot_slope_field(ax, x, y)

for (x_0, y_0) in ((0, 1), (-1, 1), (-6, 0.01)):
    x, y = logistic_fine.starting_from(x_0, y_0)
    ax.plot(x, y)

Beautiful!

Let's look at how our results depend on the step size $\Delta x$.  We can see that if we want to integrate from $x=0$ to $x=10$ we will need more and more steps as we use smaller and smaller $\Delta x$s.

In [ ]:
k, L = 1, 2

logistic = LogisticODE(k, L)

# Both of these integrators 'cover the same distance'
logistic_coarse = EulerMethod(logistic, 1,     10   ) # but this one takes big steps
logistic_medium = EulerMethod(logistic, 0.1,   100  ) # and this one takes smaller steps!

fig, ax = plt.subplots()

x = np.linspace(0, 10, 20)
y = np.linspace(1, 2,  20)
logistic.plot_slope_field(ax, x, y)

for integrator in (logistic_coarse, logistic_medium, logistic_fine):
    x, y = integrator.starting_from(0, 1)
    ax.plot(x, y, label=f'Δx = {integrator.Delta_x}')

ax.legend()
ax.set_xlabel('x')
ax.set_ylabel('y')

You can see that when $\Delta x$ is big we take big chunky steps and get a very blocky function that you know must be unrealistic---functions that obey differential equations usually don't have such dramatic cusps.

You can also see that when we make $\Delta x$ smaller the resulting function gets much smoother and stops changing too much---the numerical results with $\Delta x = 0.1$ and $\Delta x = 0.001$ hardly differ (at least by eye).

In [ ]:
k, L = 1, 2
logistic = LogisticODE(k, L)

fig, ax = plt.subplots()

destination = 5
x = np.linspace(0, destination, 20)
y = np.linspace(1, 2, 20)
logistic.plot_slope_field(ax, x, y)

for power in np.arange(0, 6):
    Delta_x = 1 / 2** power
    steps = int(destination//Delta_x)
    integrator = EulerMethod(logistic, Delta_x, steps)
    x, y = integrator.starting_from(0, 1)
    ax.plot(x, y, label=f'Δx = 2^-{power}')

ax.legend()

# How the Error Behaves

This numerical approach to solving differential equations to produce functions numerically requires a very different attitude.  We accept that we will make small mistakes because our $\Delta x$ isn't infinitesimally small.  

Of course the mistake at the end of the calculation gets smaller and smaller as $\Delta x$ gets smaller.  But an economic question: *how fast does the mistake get smaller*?

Euler's method is called a *first order method* because the mistakes that it makes at each step are $\Delta x^2$ or smaller (meaning higher power of $\Delta x$).

If each step's mistake is $\Delta x^2$ and we integrate over a range of $x$ then the *number of steps* is $x / \Delta x$.  The errors from each step can accumulate, and so by the end our mistake could be like (number of steps)$\times$(mistake at each step) $\sim 1/\Delta x \times \Delta x^2 = \Delta x$.  The total error isn't *equal* to $\Delta x$ but it *scales like* $\Delta x$.

Let's check that scaling now.  Suppose we want to integrate from $(x_0, y_0) = (0, 1)$ to find $y(x=1)$ using smaller and smaller step sizes $\Delta x$.

In [ ]:
for i, letter in enumerate(('a', 'b', 'c')):
    print(i, letter)

In [ ]:
DELTA_X = 1.0/2**np.arange(0, 15)
Y_AT_END = np.zeros_like(DELTA_X)

fig, ax = plt.subplots()
logistic.plot_slope_field(ax, np.linspace(0, 1, 20), np.linspace(1, 2, 20))

for i, Delta_x in enumerate(DELTA_X):
    integrator = EulerMethod(logistic, Delta_x, int(1/Delta_x))
    x, y = integrator.starting_from(0, 1)
    ax.plot(x, y, label=f'Δx={Delta_x}')
    Y_AT_END[i] = y[-1]

ax.plot(x, logistic.exact_solution(x), label='exact solution')
ax.legend()

exact_value = logistic.exact_solution(1)
error_at_the_end = np.abs(Y_AT_END - exact_value)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(16,5))
fig.suptitle('Error after integrating to x=1 from (x_0, y_0) = (0, 1)')

for a in ax:
    a.plot(DELTA_X, error_at_the_end, marker='o', linestyle='none')
    a.set_xlabel('Δx')
    a.set_ylabel('Error = |numerical - exact|')

ax[1].set_xscale('log')
ax[1].set_yscale('log')

Looking at either figure leads you to the conclusion that $\text{error} \sim \Delta x$.  In other words, the relationship is a line and it goes through 0, $\text{error} = m \times \Delta x$ for some $m$.  We don't *really* are about the slope that much---it's determined by the differential equation itself---but the *scaling* is determined by Euler's method.